# GameTheory 2e — IESDS et rationalisabilité : élaguer un jeu, puis mesurer ce qui reste (C#)

**Navigation** : [<< 2-NormalForm (track principal)](GameTheory-02-NormalForm-Python.ipynb) · [2e-Python (jumeau)](GameTheory-02e-IESDS-Rationalisation-Python.ipynb) | [Index](README.md) | [4-NashEquilibrium >>](GameTheory-04-NashEquilibrium-Python.ipynb)

**Kernel** : `.net-csharp` (.NET Interactive) — aucun appel réseau, aucun package externe.

**Twin C# (.NET Interactive)** de [GameTheory-02e-IESDS-Rationalisation-Python.ipynb](GameTheory-02e-IESDS-Rationalisation-Python.ipynb) — parité .NET ⇄ Python. Le jumeau Python s'appuie sur `numpy` et `scipy.optimize.linprog` ; ce carnet réimplémente les mêmes algorithmes **from scratch** en C# : dominance, IESDS, énumération de sommets du polyèdre de faisabilité, et un **simplex à deux phases** maison qui remplace `linprog`. Les familles exhaustives (4096 jeux 2×3) et tous les compteurs déterministes reproduisent les mêmes mesures que le jumeau Python.

## Pourquoi ce carnet existe

La série pose le jeu en forme normale (`GameTheory-02-NormalForm`), enchaîne sur la topologie des jeux 2×2 (`GameTheory-03-Topology2x2`) puis arrive directement à l'équilibre de Nash (`GameTheory-04-NashEquilibrium`). Entre les deux, il manque **le concept de solution qui les relie** : la **rationalisabilité** — et, en pratique, l'**élimination itérée des stratégies strictement dominées** (IESDS).

**Ce que le carnet 02 fait déjà** — on ne le refait pas : il définit la dominance stricte et implémente une IESDS sur un jeu 3×3.

**Ce qu'il ne fait pas** — et que ce carnet fait :

1. justifier que le résultat de l'IESDS **ne dépend pas de l'ordre** d'élimination (théorème pour la dominance stricte, **contre-exemple mesuré** pour la dominance faible) ;
2. distinguer l'IESDS (éliminer ce qui est *dominé*) de l'**admissibilité** (éliminer ce qui n'est *jamais meilleure réponse*) ;
3. relier les deux notions de solution par l'inclusion `rationalisable ⊇ Nash`, et exhiber un profil rationalisable qui **n'est pas** un équilibre de Nash.

## Plan

1. Briques : le jeu bimatriciel local, et les fonctions de dominance
2. L'IESDS pas-à-pas, avec son journal d'éliminations
3. L'indépendance à l'ordre : mesure exhaustive sur 4096 jeux 2×3, et contre-exemple en dominance faible
4. L'admissibilité et les stratégies rationalisables
5. `Nash ⊆ rationalisable` : la mesure exhaustive, et un exemple de divergence
6. Exercices

## Prérequis

`GameTheory-02-NormalForm` (forme normale, dominance, meilleure réponse) et, pour la version formelle des mêmes notions, `GameTheory-02b-Lean-Definitions-Lean`. L'équilibre de Nash lui-même est repris de `GameTheory-04-NashEquilibrium` : ici on ne fait que le comparer aux ensembles rationalisables.

## Références

**Osborne & Rubinstein (1994)**, *A Course in Game Theory*, ch. 4 « Rationalizability » (correlated rationalizability, élimination itérée) et ch. 2 (dominance et élimination des stratégies dominées). **Bernheim (1984)** et **Pearce (1984)**, « Rationalizable strategic behavior » / « ... and the problem of perfection », *Econometrica* 52(4) — les deux textes fondateurs de la rationalisabilité, et le lemme de Pearce (deux joueurs : jamais-meilleure-réponse à une croyance mixte = strictement dominée par un mélange). **Leyton-Brown & Shoham (2008)**, *Essentials of Game Theory*, ch. 2 (dominance, meilleure réponse, équilibre). Gisement cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`.

In [1]:
// Helpers, representation du jeu bimatriciel, et recherche des Nash purs.
// Jumeau C# des briques locales : le Python importe NormalFormGame/find_pure_nash
// du module partage game_theory_utils.py ; ici les memes objets, from scratch.
using System;
using System.Linq;
using System.Text;
using System.Globalization;
using System.Collections.Generic;

// Culture invariante : les nombres s'affichent avec un point decimal, quelle
// que soit la locale de la machine -- parite avec les sorties du jumeau Python.
public static readonly CultureInfo INV = CultureInfo.InvariantCulture;

static string FI(double x, string fmt = "F1") => x.ToString(fmt, INV);

// Affiche une matrice ligne par ligne (comme le numpy du jumeau Python).
public static void PrintMat(double[,] X)
{
    int m = X.GetLength(0), n = X.GetLength(1);
    for (int i = 0; i < m; i++)
        Console.WriteLine("  [" + string.Join("  ", Enumerable.Range(0, n).Select(j => X[i, j].ToString("0.##", INV))) + "]");
}

// Jeu bimatriciel (forme normale), m strategies Ligne x n strategies Colonne.
public class Game
{
    public double[,] A { get; }
    public double[,] B { get; }
    public string[] RowLabels { get; }
    public string[] ColLabels { get; }
    public string Name { get; }
    public int M => A.GetLength(0);
    public int N => A.GetLength(1);

    public Game(double[,] a, double[,] b, string[] rowLabels = null,
                string[] colLabels = null, string name = "Game")
    {
        A = a; B = b;
        RowLabels = rowLabels ?? DefaultLabels(M, "R");
        ColLabels = colLabels ?? DefaultLabels(N, "C");
        Name = name;
    }

    static string[] DefaultLabels(int n, string p) =>
        Enumerable.Range(0, n).Select(i => p + i).ToArray();

    // Affichage : lignes "( a, b)" alignees, comme NormalFormGame.display() en Python.
    public void Display()
    {
        int wl = RowLabels.Max(s => s.Length);
        var sb = new StringBuilder();
        sb.AppendLine();
        sb.AppendLine(Name);
        sb.AppendLine(new string('=', 50));
        sb.Append(new string(' ', wl + 3));
        foreach (var c in ColLabels) sb.Append(string.Format("{0,12}", c));
        sb.AppendLine();
        sb.AppendLine(new string('-', wl + 3 + 12 * N));
        for (int i = 0; i < M; i++)
        {
            sb.Append(string.Format(INV, "{0," + wl + "}  ", RowLabels[i]));
            for (int j = 0; j < N; j++)
                sb.Append(string.Format(INV, "({0,4},{1,4})  ", A[i, j].ToString("F1", INV), B[i, j].ToString("F1", INV)));
            sb.AppendLine();
        }
        Console.Write(sb.ToString());
    }
}

// Equilibres de Nash purs : (i, j) tel que A[i,j] soit maximal dans sa colonne
// (meilleure reponse de la Ligne) ET B[i,j] maximal dans sa ligne (Colonne).
// Jumeau de find_pure_nash -- egalites admises, comme en Python.
public static List<(int i, int j)> FindPureNash(Game g)
{
    var res = new List<(int, int)>();
    for (int i = 0; i < g.M; i++)
        for (int j = 0; j < g.N; j++)
        {
            bool rowBest = true, colBest = true;
            for (int k = 0; k < g.M; k++) if (g.A[k, j] > g.A[i, j] + 1e-9) rowBest = false;
            for (int l = 0; l < g.N; l++) if (g.B[i, l] > g.B[i, j] + 1e-9) colBest = false;
            if (rowBest && colBest) res.Add((i, j));
        }
    return res;
}

Console.WriteLine("Briques chargees : Game (bimatriciel), FindPureNash, PrintMat.");
var jeuDemo = new Game(new double[,] { { 3, 0 }, { 5, 1 } }, new double[,] { { 3, 5 }, { 0, 1 } },
                       new[] { "Cooperer", "Defaire" }, new[] { "Cooperer", "Defaire" },
                       "Dilemme du Prisonnier (verification d'affichage)");
jeuDemo.Display();
Console.WriteLine("Nash purs : " + string.Join(", ", FindPureNash(jeuDemo).Select(p => "(" + p.i + "," + p.j + ")")));

The below script needs to be able to find the current output cell; this is an easy method to get it.

Briques chargees : Game (bimatriciel), FindPureNash, PrintMat.



Dilemme du Prisonnier (verification d'affichage)
               Cooperer     Defaire
-----------------------------------
Cooperer  ( 3.0, 3.0)  ( 0.0, 5.0)  
 Defaire  ( 5.0, 0.0)  ( 1.0, 1.0)  


Nash purs : (1,1)


## 1. La dominance stricte, et ce qu'elle ne dit pas

Une stratégie $s_i$ **domine strictement** une stratégie $s_i'$ si le joueur $i$ fait **strictement mieux** avec $s_i$ qu'avec $s_i'$, **quelle que soit** la stratégie des autres :

$$u_i(s_i, s_{-i}) > u_i(s_i', s_{-i}) \quad \forall s_{-i} \in S_{-i}$$

Deux points qu'il faut garder en tête avant d'en déduire :

- la dominance est **stricte** : si l'inégalité est partout large et égale quelque part, on parle de dominance *faible* — et la section 3 montrera que les deux ne se comportent pas du tout pareil ;
- la dominance se lit **joueur par joueur** : une stratégie de la Ligne ne peut être éliminée que si *la Ligne* est mieux servie par une autre de ses stratégies.

En Python, `NormalFormGame` et `find_pure_nash` viennent du module partagé `game_theory_utils.py` ; en C#, les briques équivalentes sont **locales** (cellule précédente). Les fonctions de dominance ci-dessous sont **nouvelles** dans les deux langages : ni le module partagé ni la classe `Game` ne les contiennent.

In [2]:
// Fonctions de dominance stricte, orientees par joueur.
// Renvoient, pour chaque strategie du joueur, l'indice de celle qui la domine
// strictement, ou null (None en Python) si rien ne la domine.
public static int?[] DominanceStricteLignes(double[,] A)
{
    int m = A.GetLength(0), n = A.GetLength(1);
    var dominateurs = new int?[m];
    for (int i = 0; i < m; i++)
    {
        dominateurs[i] = null;
        for (int k = 0; k < m; k++)
        {
            if (k == i) continue;
            bool batTout = true;
            for (int j = 0; j < n; j++) if (!(A[k, j] > A[i, j])) batTout = false;
            if (batTout) { dominateurs[i] = k; break; }
        }
    }
    return dominateurs;
}

// Pour le joueur Colonne les strategies sont les colonnes : B[i, l] > B[i, j]
// sur toutes les lignes i, pour une autre colonne l.
public static int?[] DominanceStricteColonnes(double[,] B)
{
    int m = B.GetLength(0), n = B.GetLength(1);
    var dominateurs = new int?[n];
    for (int j = 0; j < n; j++)
    {
        dominateurs[j] = null;
        for (int l = 0; l < n; l++)
        {
            if (l == j) continue;
            bool batTout = true;
            for (int i = 0; i < m; i++) if (!(B[i, l] > B[i, j])) batTout = false;
            if (batTout) { dominateurs[j] = l; break; }
        }
    }
    return dominateurs;
}

// Verification sur un jeu canonique : le Dilemme du Prisonnier
double[,] IPD_A = { { 3, 0 }, { 5, 1 } };
double[,] IPD_B = { { 3, 5 }, { 0, 1 } };

var domLignes = DominanceStricteLignes(IPD_A);
var domColonnes = DominanceStricteColonnes(IPD_B);
Console.WriteLine("Ligne 0 (Cooperer) dominee par : " + (domLignes[0]?.ToString() ?? "None"));
Console.WriteLine("Ligne 1 (Defaire) dominee par : " + (domLignes[1]?.ToString() ?? "None"));
Console.WriteLine("Colonne 0 (Cooperer) dominee par : " + (domColonnes[0]?.ToString() ?? "None"));
Console.WriteLine("Colonne 1 (Defaire) dominee par : " + (domColonnes[1]?.ToString() ?? "None"));

Ligne 0 (Cooperer) dominee par : 1


Ligne 1 (Defaire) dominee par : None


Colonne 0 (Cooperer) dominee par : 1


Colonne 1 (Defaire) dominee par : None


### Lecture du résultat

Les quatre lignes affichent `1` ou `None` : `Cooperer` est strictement dominée par `Defaire` (5 > 3 et 1 > 0), chez la Ligne comme chez la Colonne, tandis que `Defaire` n'est dominée par rien. Le Dilemme du Prisonnier a donc, pour chaque joueur, une stratégie strictement dominée (`Cooperer`) et une stratégie indomptable (`Defaire`) — c'est exactement ce que l'IESDS va énumérer, étape par étape.

## 2. L'IESDS pas-à-pas

La **procédure** IESDS est la suivante : on cherche une stratégie strictement dominée, on la retire du jeu, on recommence sur le jeu réduit, jusqu'à ce qu'aucune stratégie ne soit plus dominée.

Deux points de méthode que la procédure cache :

- l'**ordre** dans lequel on retire les stratégies dominées (section 3) ;
- le fait qu'**éliminer une stratégie peut en créer une nouvelle** : sur le jeu réduit, une stratégie qui n'était pas dominée peut le devenir, parce que la stratégie qui la dominait a disparu.

Le code ci-dessous tient un **journal** des éliminations : chaque étape indique le joueur, la stratégie retirée et la stratégie qui la dominait. Le paramètre `ordre` reçoit des couples `('L', i)` / `('C', j)` aux indices **originaux** des stratégies : la priorité d'examen reste stable quand le jeu se réduit.

In [3]:
// L'IESDS pas-a-pas, avec journal. Toutes les comparaisons se font sur la
// sous-matrice des strategies actives (le jeu REDUIT), en indexant la matrice
// originale par les listes d'indices actifs -- equivalent au np.delete du Python.
public static (List<string> survL, List<string> survC, List<string> journal,
               double[,] Ared, double[,] Bred)
    Iesds(double[,] A, double[,] B, string[] labelsL = null, string[] labelsC = null,
          List<(char joueur, int index)> ordre = null)
{
    int m = A.GetLength(0), n = A.GetLength(1);
    var tousL = labelsL ?? Enumerable.Range(0, m).Select(i => "R" + i).ToArray();
    var tousC = labelsC ?? Enumerable.Range(0, n).Select(j => "C" + j).ToArray();
    var actifsL = Enumerable.Range(0, m).ToList();
    var actifsC = Enumerable.Range(0, n).ToList();

    if (ordre == null)
        ordre = Enumerable.Range(0, m).Select(i => ('L', i))
                .Concat(Enumerable.Range(0, n).Select(j => ('C', j))).ToList();

    var journal = new List<string>();
    int etape = 0;
    while (true)
    {
        bool elimine = false;
        foreach (var (joueur, index) in ordre)
        {
            if (joueur == 'L')
            {
                if (!actifsL.Contains(index)) continue;
                int pos = actifsL.IndexOf(index);   // position, pour RemoveAt seulement
                int? dominante = null;
                foreach (var k in actifsL)
                {
                    if (k == index) continue;
                    bool batTout = true;
                    foreach (var j in actifsC)
                        if (!(A[k, j] > A[index, j])) batTout = false;   // indices ORIGINAUX
                    if (batTout) { dominante = k; break; }
                }
                if (dominante.HasValue)
                {
                    etape++;
                    journal.Add("Etape " + etape + " : Ligne elimine " + tousL[index]
                                + ", strictement dominee par " + tousL[dominante.Value]);
                    actifsL.RemoveAt(pos);
                    elimine = true;
                    break;
                }
            }
            else
            {
                if (!actifsC.Contains(index)) continue;
                int pos = actifsC.IndexOf(index);   // position, pour RemoveAt seulement
                int? dominante = null;
                foreach (var l in actifsC)
                {
                    if (l == index) continue;
                    bool batTout = true;
                    foreach (var i in actifsL)
                        if (!(B[i, l] > B[i, index])) batTout = false;   // indices ORIGINAUX
                    if (batTout) { dominante = l; break; }
                }
                if (dominante.HasValue)
                {
                    etape++;
                    journal.Add("Etape " + etape + " : Colonne elimine " + tousC[index]
                                + ", strictement dominee par " + tousC[dominante.Value]);
                    actifsC.RemoveAt(pos);
                    elimine = true;
                    break;
                }
            }
        }
        if (!elimine) break;
    }

    var Ared = new double[actifsL.Count, actifsC.Count];
    var Bred = new double[actifsL.Count, actifsC.Count];
    for (int i = 0; i < actifsL.Count; i++)
        for (int j = 0; j < actifsC.Count; j++)
        {
            Ared[i, j] = A[actifsL[i], actifsC[j]];
            Bred[i, j] = B[actifsL[i], actifsC[j]];
        }
    return (actifsL.Select(i => tousL[i]).ToList(),
            actifsC.Select(j => tousC[j]).ToList(), journal, Ared, Bred);
}

Console.WriteLine("Iesds chargee : signature (survivants_lignes, survivants_colonnes, journal, A_reduit, B_reduit).");

Iesds chargee : signature (survivants_lignes, survivants_colonnes, journal, A_reduit, B_reduit).


### Un jeu en cascade

Le jeu ci-dessous est construit pour que l'élimination soit **en cascade** : une stratégie de la Colonne n'est dominée qu'**après** le retrait d'une stratégie de la Ligne. C'est le point pédagogique central — la dominance se calcule sur le jeu **réduit**, pas sur le jeu initial.

In [4]:
// Jeu en cascade : le retrait de R0 rend C2 strictement dominee.
double[,] A_cascade = {
    { 0, 0, 0 },   // R0
    { 1, 1, 1 },   // R1
    { 3, 0, 3 },   // R2
};
double[,] B_cascade = {
    { 0, 0, 0 },    // R0
    { 2, 1, 1 },    // R1
    { 0, 3, -1 },   // R2
};

var jeuCascade = new Game(A_cascade, B_cascade,
                          new[] { "R0", "R1", "R2" }, new[] { "C0", "C1", "C2" },
                          "Jeu en cascade");
jeuCascade.Display();

static string DomStr(int?[] doms) =>
    "{" + string.Join(", ", doms.Select((d, i) => i + ": " + (d?.ToString() ?? "None"))) + "}";

Console.WriteLine("Strategies strictement dominees AVANT toute elimination :");
Console.WriteLine("  Ligne   : " + DomStr(DominanceStricteLignes(A_cascade)));
Console.WriteLine("  Colonne : " + DomStr(DominanceStricteColonnes(B_cascade)));

var (survL, survC, journalCascade, A_red, B_red) = Iesds(A_cascade, B_cascade,
                                                         new[] { "R0", "R1", "R2" },
                                                         new[] { "C0", "C1", "C2" });
Console.WriteLine();
Console.WriteLine("Journal de l'IESDS :");
foreach (var l in journalCascade) Console.WriteLine("  " + l);
Console.WriteLine();
Console.WriteLine("Survivants : lignes [" + string.Join(", ", survL) + "] | colonnes [" + string.Join(", ", survC) + "]");
Console.WriteLine("Jeu reduit final :");
PrintMat(A_red);
PrintMat(B_red);


Jeu en cascade
               C0          C1          C2
-----------------------------------------
R0  ( 0.0, 0.0)  ( 0.0, 0.0)  ( 0.0, 0.0)  
R1  ( 1.0, 2.0)  ( 1.0, 1.0)  ( 1.0, 1.0)  
R2  ( 3.0, 0.0)  ( 0.0, 3.0)  ( 3.0,-1.0)  


Strategies strictement dominees AVANT toute elimination :


  Ligne   : {0: 1, 1: None, 2: None}


  Colonne : {0: None, 1: None, 2: None}


Journal de l'IESDS :


...

Jeu reduit final :


  [1  1]


  [3  0]


  [2  1]


  [0  3]


### Lecture du résultat

Avant toute élimination, **seule** `R0` est strictement dominée (par `R1`, qui vaut `1 > 0` sur les trois colonnes) ; aucune stratégie de la Colonne ne l'est encore — `C2` vaut `[0, 1, -1]` et `C0` vaut `[0, 2, 0]` : sur la ligne `R0`, les deux sont à égalité (`0`), donc `C0` ne domine **pas** strictement `C2`.

Le journal montre la cascade en deux temps :

1. **R0 est éliminée** par `R1` ;
2. sur le jeu 2×3 restant, `C2` devient strictement dominée par `C0` (`2 > 1` sur `R1`, `0 > -1` sur `R2`) : elle est éliminée à son tour.

Le jeu final est un 2×2 (`R1`, `R2` × `C0`, `C1`). Deux remarques :

- l'**ordre** des deux éliminations n'a pas d'importance ici (elles sont de joueurs différents) — mais qu'en est-il en général ? C'est la question de la section suivante ;
- le jeu réduit **ne dit pas** quel sera l'équilibre : élaguer un jeu, ce n'est pas le résoudre.

## 3. L'indépendance à l'ordre

**Énoncé.** Pour la dominance **stricte**, l'IESDS est *indépendante de l'ordre* : quelle que soit la stratégie dominée retirée en premier, l'ensemble final des survivants est le même.

L'idée de la preuve est simple : la dominance stricte est une relation « pour tout » (sur toutes les stratégies de l'adversaire), donc elle survit au rétrécissement du jeu ; et la relation d'ordre sur les dominateurs est **stricte** (un dominateur strict ne peut pas être lui-même strictement dominé par une stratégie dominée, sans cycle de gains). Le résultat est donc un point fixe, indépendant du chemin.

**Faut-il le croire ?** Non : on le mesure. Le code ci-dessous parcourt **tous** les jeux 2×3 dont les gains sont dans `{0, 1}` — soit $2^6 \times 2^6 = 4096$ jeux — et, pour chacun, compare les survivants obtenus sous plusieurs ordres d'élimination tirés au hasard (graine fixée, donc reproductible). Le tirage utilise un **PRNG maison** (mulberry32, graine `0`) : reproductible indépendamment du runtime .NET, là où le jumeau Python utilise `numpy.random.default_rng(0)`.

In [5]:
// PRNG maison (mulberry32) : reproductible, independant du runtime .NET.
public sealed class Rng
{
    uint _s;
    public Rng(uint seed) => _s = seed;
    uint NextU()
    {
        _s += 0x6D2B79F5u;
        uint t = _s;
        t = (t ^ (t >> 15)) * t | 1u;
        t ^= t + (t ^ (t >> 7)) * (t | 61u);
        return t ^ (t >> 14);
    }
    public int NextInt(int lo, int hi) => lo + (int)(NextU() % (uint)(hi - lo));
    public void Shuffle<T>(IList<T> list)
    {
        for (int i = list.Count - 1; i > 0; i--)
        {
            int j = NextInt(0, i + 1);
            (list[i], list[j]) = (list[j], list[i]);
        }
    }
}

// Tous les jeux 2x3 dont les gains des deux joueurs sont dans {0, 1}
// (2^6 x 2^6 = 4096 jeux) -- meme enumeration que itertools.product du Python.
public static IEnumerable<(double[,] A, double[,] B)> TousLesJeux2x3()
{
    for (int aBits = 0; aBits < 64; aBits++)
        for (int bBits = 0; bBits < 64; bBits++)
        {
            var A = new double[2, 3];
            var B = new double[2, 3];
            for (int t = 0; t < 6; t++)
            {
                A[t / 3, t % 3] = (aBits >> t) & 1;
                B[t / 3, t % 3] = (bBits >> t) & 1;
            }
            yield return (A, B);
        }
}

// Ordres d'examen aleatoires des strategies (lignes et colonnes melees).
public static IEnumerable<List<(char, int)>> OrdresAleatoires(Rng rng, int m, int n, int nombre)
{
    var elements = Enumerable.Range(0, m).Select(i => ('L', i))
                    .Concat(Enumerable.Range(0, n).Select(j => ('C', j))).ToList();
    for (int t = 0; t < nombre; t++)
    {
        var melange = new List<(char, int)>(elements);
        rng.Shuffle(melange);
        yield return melange;
    }
}

// Mesure exhaustive : l'IESDS stricte depend-elle de l'ordre ?
var rng = new Rng(0);
int nombreJeux = 0;
int jeuxDivergents = 0;
foreach (var (A, B) in TousLesJeux2x3())
{
    nombreJeux++;
    var survivants = new HashSet<string>();
    foreach (var ordre in OrdresAleatoires(rng, 2, 3, 10))
    {
        var (lig, col, _, _, _) = Iesds(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" }, ordre);
        survivants.Add(string.Join(";", lig) + "|" + string.Join(";", col));
    }
    if (survivants.Count > 1) jeuxDivergents++;
}

Console.WriteLine("Jeux 2x3 a gains dans {0, 1} analyses : " + nombreJeux);
Console.WriteLine("Jeux dont les survivants dependent de l'ordre : " + jeuxDivergents);
Console.WriteLine("Ordres tires par jeu : 10 (PRNG maison mulberry32, graine 0, reproductible)");

Jeux 2x3 a gains dans {0, 1} analyses : 4096


Jeux dont les survivants dependent de l'ordre : 0


Ordres tires par jeu : 10 (PRNG maison mulberry32, graine 0, reproductible)


### Lecture du résultat

Sur les **4096** jeux 2×3 exhaustivement énumérés, avec dix ordres d'élimination tirés au hasard par jeu, **aucun** jeu ne présente de divergence : l'ensemble des survivants est indépendant de l'ordre. C'est la mesure, pas l'intuition, qui l'établit — et elle est reproductible (graine `0`).

Cette propriété est **spécifique à la dominance stricte**. Avec la dominance *faible* (inégalité large partout, stricte quelque part), l'ordre change le résultat : on le cherche ci-dessous.

### La dominance faible : un ordre qui change tout

Une stratégie $s_i$ est **faiblement dominée** par $s_i'$ si $u_i(s_i', s_{-i}) \geq u_i(s_i, s_{-i})$ pour tout $s_{-i}$, avec inégalité stricte quelque part. La différence avec le cas strict est décisive : la stratégie faiblement dominée peut être **elle-même** la dominatrice d'une autre stratégie. Supprimer la dominée peut donc supprimer sa dominatrice, et l'ordre devient significatif.

Plutôt que d'inventer un exemple à la main, on **cherche** un contre-exemple dans la même famille exhaustive — 4096 jeux 2×3 à gains dans `{0, 1}` — en comparant deux ordres opposés.

In [6]:
// Contre-exemple : l'elimination en dominance FAIBLE depend de l'ordre.
// Faiblement dominee : une autre strategie fait AUSSI BIEN partout (>=) et
// STRICTEMENT MIEUX quelque part (>).
public static (List<string> survL, List<string> survC, List<string> journal)
    IesdsFaible(double[,] A, double[,] B, string[] labelsL = null, string[] labelsC = null,
                List<(char joueur, int index)> ordre = null)
{
    int m = A.GetLength(0), n = A.GetLength(1);
    var tousL = labelsL ?? Enumerable.Range(0, m).Select(i => "R" + i).ToArray();
    var tousC = labelsC ?? Enumerable.Range(0, n).Select(j => "C" + j).ToArray();
    var actifsL = Enumerable.Range(0, m).ToList();
    var actifsC = Enumerable.Range(0, n).ToList();

    if (ordre == null)
        ordre = Enumerable.Range(0, m).Select(i => ('L', i))
                .Concat(Enumerable.Range(0, n).Select(j => ('C', j))).ToList();

    var journal = new List<string>();
    int etape = 0;
    while (true)
    {
        bool elimine = false;
        foreach (var (joueur, index) in ordre)
        {
            if (joueur == 'L')
            {
                if (!actifsL.Contains(index)) continue;
                int pos = actifsL.IndexOf(index);   // position, pour RemoveAt seulement
                int? dominante = null;
                foreach (var k in actifsL)
                {
                    if (k == index) continue;
                    bool partoutAuMoinsAussiBien = true, strictementMieuxQuelquePart = false;
                    foreach (var j in actifsC)
                    {
                        if (!(A[k, j] >= A[index, j])) partoutAuMoinsAussiBien = false;   // indices ORIGINAUX
                        if (A[k, j] > A[index, j]) strictementMieuxQuelquePart = true;
                    }
                    if (partoutAuMoinsAussiBien && strictementMieuxQuelquePart) { dominante = k; break; }
                }
                if (dominante.HasValue)
                {
                    etape++;
                    journal.Add("Etape " + etape + " : Ligne elimine " + tousL[index]
                                + " (faiblement dominee par " + tousL[dominante.Value] + ")");
                    actifsL.RemoveAt(pos);
                    elimine = true;
                    break;
                }
            }
            else
            {
                if (!actifsC.Contains(index)) continue;
                int pos = actifsC.IndexOf(index);   // position, pour RemoveAt seulement
                int? dominante = null;
                foreach (var l in actifsC)
                {
                    if (l == index) continue;
                    bool partoutAuMoinsAussiBien = true, strictementMieuxQuelquePart = false;
                    foreach (var i in actifsL)
                    {
                        if (!(B[i, l] >= B[i, index])) partoutAuMoinsAussiBien = false;   // indices ORIGINAUX
                        if (B[i, l] > B[i, index]) strictementMieuxQuelquePart = true;
                    }
                    if (partoutAuMoinsAussiBien && strictementMieuxQuelquePart) { dominante = l; break; }
                }
                if (dominante.HasValue)
                {
                    etape++;
                    journal.Add("Etape " + etape + " : Colonne elimine " + tousC[index]
                                + " (faiblement dominee par " + tousC[dominante.Value] + ")");
                    actifsC.RemoveAt(pos);
                    elimine = true;
                    break;
                }
            }
        }
        if (!elimine) break;
    }
    return (actifsL.Select(i => tousL[i]).ToList(),
            actifsC.Select(j => tousC[j]).ToList(), journal);
}

var ORDRE_A = new List<(char, int)> { ('L', 0), ('L', 1), ('C', 0), ('C', 1), ('C', 2) };
var ORDRE_B = new List<(char, int)>(ORDRE_A);
ORDRE_B.Reverse();

int nbContreExemples = 0;
double[,]? A_ex = null;
double[,]? B_ex = null;
List<string> ligA = null, colA = null, ligB = null, colB = null;
foreach (var (A, B) in TousLesJeux2x3())
{
    var (la, ca, _) = IesdsFaible(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" }, ORDRE_A);
    var (lb, cb, _) = IesdsFaible(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" }, ORDRE_B);
    if (!la.SequenceEqual(lb) || !ca.SequenceEqual(cb))
    {
        nbContreExemples++;
        if (A_ex == null) { A_ex = A; B_ex = B; ligA = la; colA = ca; ligB = lb; colB = cb; }
    }
}

Console.WriteLine("Jeux 2x3 ou l'ordre change le resultat (dominance faible) : " + nbContreExemples);

if (A_ex != null)
{
    Console.WriteLine();
    Console.WriteLine("Contre-exemple (premier trouve) :");
    Console.WriteLine("  Ordre A : L0, L1, C0, C1, C2  ->  lignes [" + string.Join(", ", ligA)
                      + "] | colonnes [" + string.Join(", ", colA) + "]");
    Console.WriteLine("  Ordre B : l'ordre inverse       ->  lignes [" + string.Join(", ", ligB)
                      + "] | colonnes [" + string.Join(", ", colB) + "]");
    Console.WriteLine();
    Console.WriteLine("  Gains de la Ligne (lignes R0, R1) :");
    for (int i = 0; i < 2; i++)
        Console.WriteLine("   R" + i + " [" + string.Join(", ", Enumerable.Range(0, 3).Select(j => A_ex[i, j].ToString("0.#", INV))) + "]");
    Console.WriteLine("  Gains de la Colonne (colonnes C0, C1, C2) :");
    for (int j = 0; j < 3; j++)
        Console.WriteLine("   C" + j + " [" + string.Join(", ", Enumerable.Range(0, 2).Select(i => B_ex[i, j].ToString("0.#", INV))) + "]");
}

Jeux 2x3 ou l'ordre change le resultat (dominance faible) : 1536


Contre-exemple (premier trouve) :


  Ordre A : L0, L1, C0, C1, C2  ->  lignes [R0] | colonnes [C1]


  Ordre B : l'ordre inverse       ->  lignes [R0, R1] | colonnes [C1]


  Gains de la Ligne (lignes R0, R1) :


...

  Gains de la Colonne (colonnes C0, C1, C2) :


   C0 [0, 0]


   C1 [1, 0]


   C2 [0, 0]



(89,10): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(90,10): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Lecture du résultat

Le compteur **non nul** est le résultat : dans la famille exhaustive des 4096 jeux 2×3 à gains dans `{0, 1}`, il existe des jeux dont l'élimination en dominance faible **dépend de l'ordre**. Le jeu affiché le montre concrètement : les deux ordres ne laissent pas les mêmes stratégies.

La conclusion est donc double, et elle est la même des deux côtés :

- **dominance stricte** : l'ordre n'a aucune importance (mesuré sur 4096 jeux, section précédente) ;
- **dominance faible** : l'ordre a de l'importance (contre-exemple mesuré, ci-dessus).

C'est la raison pour laquelle l'IESDS — et non l'élimination en dominance faible — est la procédure canonique associée aux concepts de solution de la série : elle est **confluente**, donc elle définit un objet (le jeu réduit) et non une procédure.

## 4. Admissibilité et rationalisabilité : la croyance pure ne suffit pas

L'IESDS élimine ce qui est *dominé*. Deux autres notions éliminent ce qui n'est **jamais une meilleure réponse** — et il faut les distinguer, car elles ne coïncident pas.

Une stratégie de la Ligne est **rationalisable au sens ponctuel** si elle est la meilleure réponse (à égalité près) à *au moins une* stratégie **pure** de la Colonne qui survit au processus. On parle de croyance **ponctuelle** : la croyance de l'adversaire est un point, pas une distribution.

Une stratégie est **rationalisable** si elle est la meilleure réponse (à égalité près) à *au moins une croyance **mixte*** — c'est-à-dire à une distribution de probabilités sur les stratégies survivantes de l'adversaire. C'est la définition de Bernheim et Pearce.

L'implication ne joue que dans un sens :

$$\text{rationalisable ponctuellement} \;\Longrightarrow\; \text{rationalisable}$$

Une croyance pure est une croyance mixte dégénérée : qui est la meilleure réponse à la stratégie pure $j$ est la meilleure réponse à la croyance qui met toute la probabilité sur $j$. La réciproque est **fausse** — et c'est le cœur de cette section.

> **Avertissement de vocabulaire.** Le mot *admissible* a un sens **standard** en théorie des jeux : une stratégie est *admissible* si elle n'est **pas faiblement dominée**, ce qui désigne un ensemble **plus grand** que les deux notions ci-dessus. L'employer pour le filtre « meilleure réponse à une croyance pure » exposerait à une confusion : c'est pourquoi le carnet parle de **rationalisabilité ponctuelle**. La fonction `StrategiesAdmissibles` (nom historique, hérité du jumeau Python) en reste l'alias.

### Le contre-exemple : trois lignes, deux colonnes

Considérons le jeu suivant (gains de la Ligne, gains de la Colonne) :

| | C0 | C1 |
|---|---|---|
| **R0** | (3, 1) | (0, 1) |
| **R1** | (0, 1) | (3, 1) |
| **R2** | (2, 0) | (2, 0) |

Regardons **R2** :

- contre la stratégie pure `C0`, les gains de la Ligne sont `3` (R0), `0` (R1), `2` (R2) : R2 est battue sur cette colonne ;
- contre `C1`, les gains sont `0` (R0), `3` (R1), `2` (R2) : R2 est battue là aussi.

R2 n'est donc la meilleure réponse à **aucune** stratégie pure. Mais placez la Colonne sur la croyance $p = (1/2,\ 1/2)$ :

$$u(R0, p) = \tfrac{3}{2} + \tfrac{0}{2} = 1{,}5 \qquad u(R1, p) = 1{,}5 \qquad u(R2, p) = 2$$

R2 est la meilleure réponse à cette **croyance mixte**, et à aucune croyance pure. Les deux notions ne coïncident donc pas : « jamais meilleure réponse à une stratégie pure » n'implique pas « jamais meilleure réponse à un mélange ».

### La question algorithmique

Pour une stratégie $i$ de gains $u(i, j)$, être la meilleure réponse à une croyance mixte $p$ s'écrit :

$$\exists\; p \geq 0,\quad \sum_j p_j = 1,\quad \forall k:\quad u(i, p) \geq u(k, p)$$

avec $u(i, p) = \sum_j p_j\, u(i, j)$. C'est un **programme linéaire** : on maximise $\varepsilon$ sous les contraintes $(u(i) - u(k)) \cdot p \geq \varepsilon$ pour tout $k \neq i$, $\sum_j p_j = 1$ et $p \geq 0$. La stratégie $i$ est rationalisable si et seulement si la valeur optimale de $\varepsilon$ est $\geq 0$ — et cette valeur **n'est pas** toujours $\geq 0$ : elle est négative exactement quand $i$ n'est la meilleure réponse à **aucune** croyance mixte. C'est donc bien l'optimalité qui tranche, et non une tautologie. (Si l'on introduisait la contrainte $k = i$, elle vaudrait $0 \geq \varepsilon$ : l'optimum serait alors $\leq 0$ et le critère deviendrait $\varepsilon = 0$. La formulation retenue exclut donc $k = i$, et c'est elle qui donne le critère $\geq 0$.)

Où le jumeau C# diverge du Python : `scipy.optimize.linprog` n'existe pas en .NET Interactive sans package externe. Ce carnet implémente donc **son propre solveur** — un simplex à deux phases avec règle de Bland, ~80 lignes, sans dépendance — et le **confronte** à l'énumération des sommets du polyèdre de faisabilité (exacte, rapide) sur des milliers d'instances : les deux méthodes exactes doivent concorder partout.

In [7]:
// Admissibilite : elimination iteree des strategies jamais meilleure reponse
// a une strategie PURE de l'adversaire ("rationalisabilite ponctuelle").
// Une strategie est retiree si elle n'est la meilleure reponse (a egalite pres)
// a AUCUNE des strategies adverses survivantes -- comparaisons sur le jeu restreint.
public static (List<string> admL, List<string> admC)
    StrategiesAdmissibles(double[,] A, double[,] B, string[] labelsL = null, string[] labelsC = null)
{
    int m = A.GetLength(0), n = B.GetLength(1);
    var tousL = labelsL ?? Enumerable.Range(0, m).Select(i => "R" + i).ToArray();
    var tousC = labelsC ?? Enumerable.Range(0, n).Select(j => "C" + j).ToArray();
    var actifsL = Enumerable.Range(0, m).ToList();
    var actifsC = Enumerable.Range(0, n).ToList();

    bool LigneEstMr(int i, int j)   // i meilleure reponse a la colonne j (jeu restreint) ?
    {
        foreach (var p in actifsL)
            if (A[p, j] > A[i, j] + 1e-9) return false;
        return true;
    }
    bool ColonneEstMr(int j, int i) // j meilleure reponse a la ligne i (jeu restreint) ?
    {
        foreach (var q in actifsC)
            if (B[i, q] > B[i, j] + 1e-9) return false;
        return true;
    }

    while (true)
    {
        // Un joueur sans strategie ne laisse plus rien a comparer : on s'arrete
        // plutot que de vider la liste adverse.
        if (actifsL.Count == 0 || actifsC.Count == 0) break;

        var aRetirerL = actifsL.Where(p => !actifsC.Any(j => LigneEstMr(p, j))).ToList();
        var aRetirerC = actifsC.Where(q => !actifsL.Any(i => ColonneEstMr(q, i))).ToList();
        if (aRetirerL.Count == 0 && aRetirerC.Count == 0) break;

        foreach (var p in aRetirerL.OrderByDescending(x => x)) actifsL.Remove(p);
        foreach (var q in aRetirerC.OrderByDescending(x => x)) actifsC.Remove(q);
    }
    return (actifsL.Select(i => tousL[i]).ToList(), actifsC.Select(j => tousC[j]).ToList());
}

// Alias historique (nom du jumeau Python) : le filtre se lit "rationalisabilite
// ponctuelle" (croyance PURE).
public static (List<string>, List<string>) StrategiesRationalisablesPures(
    double[,] A, double[,] B, string[] labelsL = null, string[] labelsC = null)
    => StrategiesAdmissibles(A, B, labelsL, labelsC);

// La Bataille des Sexes : le classique de la divergence
double[,] A_BOS = { { 2, 0 }, { 0, 1 } };   // Opera, Ballet
double[,] B_BOS = { { 1, 0 }, { 0, 2 } };
var jeuBos = new Game(A_BOS, B_BOS, new[] { "Opera", "Ballet" }, new[] { "Opera", "Ballet" },
                      "Bataille des Sexes");
jeuBos.Display();

var nashPurs = FindPureNash(jeuBos);
var (admL, admC) = StrategiesAdmissibles(A_BOS, B_BOS, new[] { "Opera", "Ballet" }, new[] { "Opera", "Ballet" });

Console.WriteLine("Equilibres de Nash purs            : ["
    + string.Join(", ", nashPurs.Select(p => "(" + jeuBos.RowLabels[p.i] + ", " + jeuBos.ColLabels[p.j] + ")")) + "]");
Console.WriteLine("Strategies rationalisables ponctuelles (Ligne) : [" + string.Join(", ", admL) + "]");
Console.WriteLine("Strategies rationalisables ponctuelles (Colonne): [" + string.Join(", ", admC) + "]");

var profilsAdmissibles = admL.SelectMany(l => admC.Select(c => (l, c))).ToList();
var profilsNash = nashPurs.Select(p => (jeuBos.RowLabels[p.i], jeuBos.ColLabels[p.j])).ToList();
var divergents = profilsAdmissibles.Where(p => !profilsNash.Contains(p)).ToList();

Console.WriteLine();
Console.WriteLine("Profils rationalisables ponctuels : " + profilsAdmissibles.Count);
Console.WriteLine("Profils de Nash          : " + profilsNash.Count);
Console.WriteLine("Rationalisables ponctuels mais PAS de Nash : ["
    + string.Join(", ", divergents.Select(d => "(" + d.Item1 + ", " + d.Item2 + ")")) + "]");


Bataille des Sexes
                Opera      Ballet
---------------------------------
 Opera  ( 2.0, 1.0)  ( 0.0, 0.0)  
Ballet  ( 0.0, 0.0)  ( 1.0, 2.0)  


Equilibres de Nash purs            : [(Opera, Opera), (Ballet, Ballet)]


Strategies rationalisables ponctuelles (Ligne) : [Opera, Ballet]


Strategies rationalisables ponctuelles (Colonne): [Opera, Ballet]


Profils rationalisables ponctuels : 4


Profils de Nash          : 2


Rationalisables ponctuels mais PAS de Nash : [(Opera, Ballet), (Ballet, Opera)]


### Lecture du résultat

Dans la Bataille des Sexes, les **deux** stratégies de chaque joueur survivent au filtre d'admissibilité (`Opera` et `Ballet` pour chacun : chacune est meilleure réponse à la stratégie correspondante de l'adversaire), alors que seuls **deux** profils sont des équilibres de Nash : `(Opera, Opera)` et `(Ballet, Ballet)`.

Le profil `(Opera, Ballet)` est donc **admissible mais pas de Nash** — c'est la divergence recherchée. Elle a une lecture immédiate : après l'élimination, chaque joueur sait que l'adversaire peut jouer `Opera` ou `Ballet` ; mais si les deux jouent `Opera`/`Ballet`, l'un d'eux ferait mieux de changer. « Admissible » dit « je n'exclurai pas ce profil », Nash dit « personne n'y dévie ».

Attention au vocabulaire : ce filtre est l'**admissibilité** (meilleure réponse à une stratégie **pure**). La **rationalisabilité** (meilleure réponse à une croyance **mixte**) est plus permissive — sur ce jeu, elle donne les mêmes survivants, mais la section suivante montre qu'il n'en va pas toujours ainsi.

In [8]:
// ================== Rationalisabilite : croyance MIXTE ==================
// Une strategie i est rationalisable s'il existe une distribution p sur les
// strategies survivantes de l'adversaire telle que i soit la meilleure reponse
// a p :  (u(i) - u(k)) . p >= 0 pour tout k != i,  p >= 0,  sum(p) = 1.
//
// Deux implementations exactes, comme en Python :
//  - MeilleureReponseMelange   : enumeration des SOMMETS du polyedre (rapide) ;
//  - MeilleureReponseMelangeLp : programme lineaire par SIMPLEX MAISON (reference).
// La contrainte k = i est EXCLUE : elle vaudrait 0 >= 0 et, introduite dans le
// programme lineaire, forcerait eps <= 0 -- le critere deviendrait eps = 0.
const double TOL = 1e-9;

// --- Systeme lineaire carre : elimination gaussienne, pivot partiel.
//     Renvoie null si la matrice est singuliere (|pivot| < 1e-12).
public static double[] LinSolve(double[,] M, double[] rhs)
{
    int d = rhs.Length;
    var U = (double[,])M.Clone();
    var b = (double[])rhs.Clone();
    for (int c = 0; c < d; c++)
    {
        int piv = c;
        for (int r = c + 1; r < d; r++)
            if (Math.Abs(U[r, c]) > Math.Abs(U[piv, c])) piv = r;
        if (Math.Abs(U[piv, c]) < 1e-12) return null;
        if (piv != c)
        {
            for (int j = 0; j < d; j++) (U[c, j], U[piv, j]) = (U[piv, j], U[c, j]);
            (b[c], b[piv]) = (b[piv], b[c]);
        }
        for (int r = c + 1; r < d; r++)
        {
            double f = U[r, c] / U[c, c];
            for (int j = c; j < d; j++) U[r, j] -= f * U[c, j];
            b[r] -= f * b[c];
        }
    }
    var x = new double[d];
    for (int r = d - 1; r >= 0; r--)
    {
        double s = b[r];
        for (int j = r + 1; j < d; j++) s -= U[r, j] * x[j];
        x[r] = s / U[r, r];
    }
    return x;
}

// Contraintes (u(i)-u(k)).p >= 0, exprimees en coordonnees libres x :
// on elimine la derniere composante de p par la normalisation
// p = (x_0, ..., x_{d-1}, 1 - sum(x)) avec d = n - 1.
// Chaque contrainte devient une droite (coef . x = rhs) du polyedre.
public static List<(double[] coef, double rhs)> ContraintesLignes(double[,] gains, int i)
{
    int m = gains.GetLength(0), n = gains.GetLength(1);
    int d = n - 1;
    var lignes = new List<(double[], double)>();
    for (int k = 0; k < m; k++)
    {
        if (k == i) continue;
        var a = new double[n];
        for (int j = 0; j < n; j++) a[j] = gains[k, j] - gains[i, j];
        var coef = new double[d];
        for (int j = 0; j < d; j++) coef[j] = a[j] - a[d];
        lignes.Add((coef, -a[d]));
    }
    for (int j = 0; j < d; j++)
    {
        var e = new double[d]; e[j] = 1.0;
        lignes.Add((e, 0.0));                       // frontiere x_j = 0
    }
    lignes.Add((Enumerable.Repeat(1.0, d).ToArray(), 1.0));   // sum(x) = 1 (frontiere)
    return lignes;
}

// Combinaisons de k indices parmi [0, n) -- backtracking, ordre lexicographique.
public static IEnumerable<int[]> Combinaisons(int n, int k)
{
    var buf = new int[k];
    IEnumerable<int[]> Rec(int start, int depth)
    {
        if (depth == k) { yield return (int[])buf.Clone(); yield break; }
        for (int v = start; v <= n - (k - depth); v++)
        {
            buf[depth] = v;
            foreach (var c in Rec(v + 1, depth + 1)) yield return c;
        }
    }
    foreach (var c in Rec(0, 0)) yield return c;
}

// C(n, k) par formule multiplicative (long).
public static long NbCombinaisons(int n, int k)
{
    if (k < 0 || k > n) return 0;
    long r = 1;
    for (int t = 0; t < k; t++) r = r * (n - t) / (t + 1);
    return r;
}

// Sommets candidats du polyedre : intersections de d frontieres actives et
// lineairement independantes (le pivot < 1e-12 ecarte les singularites).
// Exact a toute taille de jeu -- croiser deux frontieres ne suffirait qu'a
// d = 2, c'est-a-dire aux jeux a trois colonnes au plus.
public static List<double[]> Sommets(int d, List<(double[] coef, double rhs)> lignes)
{
    var candidats = new List<double[]>();
    for (int j = 0; j < d; j++) { var e = new double[d]; e[j] = 1.0; candidats.Add(e); }
    candidats.Add(new double[d]);
    foreach (var combo in Combinaisons(lignes.Count, d))
    {
        var M = new double[d, d];
        var rhs = new double[d];
        for (int t = 0; t < d; t++)
        {
            for (int j = 0; j < d; j++) M[t, j] = lignes[combo[t]].coef[j];
            rhs[t] = lignes[combo[t]].rhs;
        }
        var x = LinSolve(M, rhs);
        if (x != null) candidats.Add(x);
    }
    return candidats;
}

// Au-dela de ce nombre de sommets candidats, l'enumeration devient trop couteuse
// et le programme lineaire (reference exacte) prend le relais.
const int SEUIL_ENUMERATION = 20000;

// ============== Simplex a deux phases, MAISON (remplace scipy.linprog) ==============
// Forme standard :  min c.z  s.t.  M z = b,  z >= 0.
// Phase 1 : variables artificielles (somme minimale > tol => infaisable).
// Phase 2 : objectif reel, artificiels verrouilles hors de la base.
// Regle de Bland (premier indice entrant, plus petit indice de base sortant
// en cas d'egalite) : terminaison garantie, pas de cycling.
static void Pivot(double[,] T, int[] basis, int r, int col, int rows, int total)
{
    double piv = T[r, col];
    for (int j = 0; j <= total; j++) T[r, j] /= piv;
    for (int r2 = 0; r2 < rows; r2++)
    {
        if (r2 == r) continue;
        double f = T[r2, col];
        if (Math.Abs(f) > 1e-15)
            for (int j = 0; j <= total; j++) T[r2, j] -= f * T[r, j];
    }
    basis[r] = col;
}

// Renvoie 0 (optimal), 2 (infaisable -- phase 1), 3 (non borne).
static int PivotLoop(double[] c, double[,] T, int[] basis, int rows, int total, int lockFrom = int.MaxValue)
{
    while (true)
    {
        var cb = new double[rows];
        for (int r = 0; r < rows; r++) cb[r] = c[basis[r]];
        int enter = -1;
        for (int j = 0; j < total && enter < 0; j++)
        {
            if (j >= lockFrom) continue;
            bool inBasis = false;
            for (int r = 0; r < rows; r++) if (basis[r] == j) inBasis = true;
            if (inBasis) continue;
            double rj = c[j];
            for (int r2 = 0; r2 < rows; r2++) rj -= cb[r2] * T[r2, j];
            if (rj < -1e-10) enter = j;            // Bland : premier indice negatif
        }
        if (enter < 0) return 0;                    // optimal
        int leave = -1;
        double best = double.PositiveInfinity;
        for (int r = 0; r < rows; r++)
        {
            if (T[r, enter] > 1e-10)
            {
                double ratio = T[r, total] / T[r, enter];
                if (ratio < best - 1e-12
                    || (Math.Abs(ratio - best) <= 1e-12 && (leave < 0 || basis[r] < basis[leave])))
                {
                    best = ratio; leave = r;
                }
            }
        }
        if (leave < 0) return 3;                    // non borne
        Pivot(T, basis, leave, enter, rows, total);
    }
}

// min cObj.z sur M z = b (b>=0 par normalisation de signe), z >= 0.
// status : 0 optimal, 2 infaisable, 3 non borne.
public static (int status, double[] z) SimplexStandard(double[] cObj, double[,] M0, double[] b0)
{
    int rows = b0.Length, cols0 = M0.GetLength(1);
    var M = (double[,])M0.Clone();
    var b = (double[])b0.Clone();
    for (int r = 0; r < rows; r++)
        if (b[r] < 0)
        {
            for (int j = 0; j < cols0; j++) M[r, j] = -M[r, j];
            b[r] = -b[r];
        }
    int total = cols0 + rows;                        // colonnes : z puis artificiels
    var T = new double[rows, total + 1];             // +1 : membre de droite
    var basis = new int[rows];
    for (int r = 0; r < rows; r++)
    {
        for (int j = 0; j < cols0; j++) T[r, j] = M[r, j];
        T[r, cols0 + r] = 1.0;
        T[r, total] = b[r];
        basis[r] = cols0 + r;
    }

    // ---- phase 1 : minimiser la somme des artificiels ----
    var c1 = new double[total];
    for (int r = 0; r < rows; r++) c1[cols0 + r] = 1.0;
    int st1 = PivotLoop(c1, T, basis, rows, total);
    double val1 = 0;
    for (int r = 0; r < rows; r++) if (basis[r] >= cols0) val1 += T[r, total];
    if (st1 != 0 || val1 > 1e-8) return (2, new double[cols0]);

    // chasser les artificiels restants (base degeneree a valeur 0) :
    // pivoter sur n'importe quelle colonne reelle non nulle de leur ligne ;
    // ligne entierement nulle = contrainte redondante, on la garde a zero.
    for (int r = 0; r < rows; r++)
    {
        if (basis[r] >= cols0)
        {
            int pivCol = -1;
            for (int j = 0; j < cols0; j++)
                if (Math.Abs(T[r, j]) > 1e-9) { pivCol = j; break; }
            if (pivCol >= 0) Pivot(T, basis, r, pivCol, rows, total);
        }
    }

    // ---- phase 2 : minimiser cObj, artificiels interdits d'entrer ----
    var c2 = new double[total];
    for (int j = 0; j < cols0; j++) c2[j] = cObj[j];
    int st2 = PivotLoop(c2, T, basis, rows, total, lockFrom: cols0);
    if (st2 != 0) return (st2, new double[cols0]);

    var z = new double[cols0];
    for (int r = 0; r < rows; r++) if (basis[r] < cols0) z[basis[r]] = T[r, total];
    return (0, z);
}

// Interface facon linprog : MAXIMISER c.x sous A_ub.x <= b_ub, A_eq.x = b_eq,
// x_j >= 0 pour j < n-1 -- la DERNIERE variable (eps) est LIBRE et se scinde
// en e+ - e-. Renvoie (status, x) avec x reconstitue ; status 0 = optimum.
public static (int status, double[] x) LpMax(double[] c, double[,] A_ub, double[] b_ub,
                                            double[,] A_eq, double[] b_eq)
{
    int nOrig = c.Length, nFixed = nOrig - 1;
    int N = nFixed + 2;                              // colonnes : x..., e+, e-
    int m1 = A_ub != null ? A_ub.GetLength(0) : 0;
    int m2 = A_eq != null ? A_eq.GetLength(0) : 0;
    int rows = m1 + m2;
    int cols0 = N + m1;                              // + slacks des inegalites
    var M = new double[rows, cols0];
    var b = new double[rows];
    for (int r = 0; r < m1; r++)
    {
        for (int j = 0; j < nFixed; j++) M[r, j] = A_ub[r, j];
        M[r, nFixed] = A_ub[r, nOrig - 1];           // +eps
        M[r, nFixed + 1] = -A_ub[r, nOrig - 1];      // -eps
        M[r, N + r] = 1.0;                           // slack
        b[r] = b_ub[r];
    }
    for (int r = 0; r < m2; r++)
    {
        for (int j = 0; j < nFixed; j++) M[m1 + r, j] = A_eq[r, j];
        M[m1 + r, nFixed] = A_eq[r, nOrig - 1];
        M[m1 + r, nFixed + 1] = -A_eq[r, nOrig - 1];
        b[m1 + r] = b_eq[r];
    }
    // maximiser c.x  <=>  minimiser -c.x
    var cMin = new double[cols0];
    for (int j = 0; j < nFixed; j++) cMin[j] = -c[j];
    cMin[nFixed] = -c[nOrig - 1];
    cMin[nFixed + 1] = c[nOrig - 1];
    var (status, zz) = SimplexStandard(cMin, M, b);
    var x = new double[nOrig];
    for (int j = 0; j < nFixed; j++) x[j] = zz[j];
    x[nOrig - 1] = zz[nFixed] - zz[nFixed + 1];
    return (status, x);
}

// Reference LP : i meilleure reponse a une croyance mixte ?
// max eps  s.t.  (u(k)-u(i)).p + eps <= 0 (k != i),  sum(p) = 1,  p >= 0.
// La valeur optimale peut etre NEGATIVE : elle l'est quand i n'est la
// meilleure reponse a aucune croyance mixte.
public static bool MeilleureReponseMelangeLp(double[,] gains, int i)
{
    int m = gains.GetLength(0), n = gains.GetLength(1);
    int nVars = n + 1;
    var c = new double[nVars]; c[nVars - 1] = 1.0;
    var A_ub = new double[m - 1, nVars];
    var b_ub = new double[m - 1];
    int t = 0;
    for (int k = 0; k < m; k++)
    {
        if (k == i) continue;
        for (int j = 0; j < n; j++) A_ub[t, j] = gains[k, j] - gains[i, j];
        A_ub[t, n] = 1.0;
        b_ub[t] = 0.0;
        t++;
    }
    var A_eq = new double[1, nVars];
    for (int j = 0; j < n; j++) A_eq[0, j] = 1.0;
    var (status, x) = LpMax(c, A_ub, b_ub, A_eq, new[] { 1.0 });
    return status == 0 && x[nVars - 1] >= -TOL;
}

// i est-elle la meilleure reponse a UNE croyance mixte ? (enumeration, exacte)
public static bool MeilleureReponseMelange(double[,] gains, int i)
{
    int m = gains.GetLength(0), n = gains.GetLength(1);
    if (n == 1)
    {
        // Croyance unique : i doit battre (ou egaler) toutes les autres.
        for (int k = 0; k < m; k++) if (gains[k, 0] > gains[i, 0] + TOL) return false;
        return true;
    }
    int d = n - 1;
    var lignes = ContraintesLignes(gains, i);
    if (NbCombinaisons(lignes.Count, d) > SEUIL_ENUMERATION)
        return MeilleureReponseMelangeLp(gains, i);
    foreach (var x in Sommets(d, lignes))
    {
        var p = new double[n];
        double s = 0;
        for (int j = 0; j < d; j++) { p[j] = x[j]; s += x[j]; }
        p[d] = 1.0 - s;
        if (p.Min() < -TOL) continue;
        bool ok = true;
        for (int k = 0; k < m && ok; k++)
        {
            double dp = 0;
            for (int j = 0; j < n; j++) dp += (gains[i, j] - gains[k, j]) * p[j];
            if (dp < -TOL) ok = false;
        }
        if (ok) return true;
    }
    return false;
}

// Sous-matrice et transposee (equivalents de np.ix_ / .T du Python).
public static double[,] SubMatrix(double[,] X, List<int> rows, List<int> cols)
{
    var S = new double[rows.Count, cols.Count];
    for (int i = 0; i < rows.Count; i++)
        for (int j = 0; j < cols.Count; j++) S[i, j] = X[rows[i], cols[j]];
    return S;
}
public static double[,] Transpose(double[,] X)
{
    int m = X.GetLength(0), n = X.GetLength(1);
    var T = new double[n, m];
    for (int i = 0; i < m; i++) for (int j = 0; j < n; j++) T[j, i] = X[i, j];
    return T;
}

// Elimination iteree des strategies jamais meilleure reponse a une croyance
// MIXTE (Bernheim, Pearce) -- jeu restreint a chaque tour.
public static (List<string> ratL, List<string> ratC)
    StrategiesRationalisables(double[,] A, double[,] B, string[] labelsL = null, string[] labelsC = null)
{
    int m = A.GetLength(0), n = B.GetLength(1);
    var tousL = labelsL ?? Enumerable.Range(0, m).Select(i => "R" + i).ToArray();
    var tousC = labelsC ?? Enumerable.Range(0, n).Select(j => "C" + j).ToArray();
    var actifsL = Enumerable.Range(0, m).ToList();
    var actifsC = Enumerable.Range(0, n).ToList();

    while (true)
    {
        if (actifsL.Count == 0 || actifsC.Count == 0) break;
        var Ar = SubMatrix(A, actifsL, actifsC);
        var Br = Transpose(SubMatrix(B, actifsL, actifsC));   // lignes = colonnes du jeu

        var aRetirerL = Enumerable.Range(0, actifsL.Count)
            .Where(p => !MeilleureReponseMelange(Ar, p)).ToList();
        var aRetirerC = Enumerable.Range(0, actifsC.Count)
            .Where(q => !MeilleureReponseMelange(Br, q)).ToList();
        if (aRetirerL.Count == 0 && aRetirerC.Count == 0) break;

        foreach (var p in aRetirerL.OrderByDescending(x => x)) actifsL.RemoveAt(p);
        foreach (var q in aRetirerC.OrderByDescending(x => x)) actifsC.RemoveAt(q);
    }
    return (actifsL.Select(i => tousL[i]).ToList(), actifsC.Select(j => tousC[j]).ToList());
}

// Le lemme de Pearce : la meme chose, par la dominance stricte MIXTE.
// i est-elle strictement dominee par un MELANGE des strategies ?
// max eps  s.t.  sum_k q_k u(k, j) - u(i, j) >= eps pour tout j ; sum(q) = 1.
public static bool StrictementDominanteMixte(double[,] gains, int i)
{
    int m = gains.GetLength(0), n = gains.GetLength(1);
    int nVars = m + 1;
    var c = new double[nVars]; c[nVars - 1] = 1.0;
    var A_ub = new double[n, nVars];
    var b_ub = new double[n];
    for (int j = 0; j < n; j++)
    {
        for (int k = 0; k < m; k++) A_ub[j, k] = -gains[k, j];
        A_ub[j, m] = 1.0;
        b_ub[j] = -gains[i, j];
    }
    var A_eq = new double[1, nVars];
    for (int k = 0; k < m; k++) A_eq[0, k] = 1.0;
    var (status, x) = LpMax(c, A_ub, b_ub, A_eq, new[] { 1.0 });
    return status == 0 && x[nVars - 1] > 1e-9;
}

// Elimination iteree des strategies strictement dominees par un melange.
public static (List<string> domL, List<string> domC)
    EliminationDominanceMixte(double[,] A, double[,] B, string[] labelsL = null, string[] labelsC = null)
{
    int m = A.GetLength(0), n = B.GetLength(1);
    var tousL = labelsL ?? Enumerable.Range(0, m).Select(i => "R" + i).ToArray();
    var tousC = labelsC ?? Enumerable.Range(0, n).Select(j => "C" + j).ToArray();
    var actifsL = Enumerable.Range(0, m).ToList();
    var actifsC = Enumerable.Range(0, n).ToList();

    while (true)
    {
        if (actifsL.Count == 0 || actifsC.Count == 0) break;
        var Ar = SubMatrix(A, actifsL, actifsC);
        var Br = Transpose(SubMatrix(B, actifsL, actifsC));

        var aRetirerL = Enumerable.Range(0, actifsL.Count)
            .Where(p => StrictementDominanteMixte(Ar, p)).ToList();
        var aRetirerC = Enumerable.Range(0, actifsC.Count)
            .Where(q => StrictementDominanteMixte(Br, q)).ToList();
        if (aRetirerL.Count == 0 && aRetirerC.Count == 0) break;

        foreach (var p in aRetirerL.OrderByDescending(x => x)) actifsL.RemoveAt(p);
        foreach (var q in aRetirerC.OrderByDescending(x => x)) actifsC.RemoveAt(q);
    }
    return (actifsL.Select(i => tousL[i]).ToList(), actifsC.Select(j => tousC[j]).ToList());
}

// --- Le contre-exemple de la section 4 : trois lignes, deux colonnes -----
double[,] A_TMB = { { 3, 0 }, { 0, 3 }, { 2, 2 } };
double[,] B_TMB = { { 1, 1 }, { 1, 1 }, { 0, 0 } };
var jeuTmb = new Game(A_TMB, B_TMB, new[] { "R0", "R1", "R2" }, new[] { "C0", "C1" },
                      "Trois lignes, deux colonnes");
jeuTmb.Display();

var pureTmb = new bool[3];
for (int i = 0; i < 3; i++)
    for (int j = 0; j < 2; j++)
    {
        bool mr = true;
        for (int k = 0; k < 3; k++) if (A_TMB[k, j] > A_TMB[i, j] + TOL) mr = false;
        if (mr) pureTmb[i] = true;
    }
var mixteTmb = Enumerable.Range(0, 3).Select(i => MeilleureReponseMelange(A_TMB, i)).ToArray();
var gainsMilieu = Enumerable.Range(0, 3)
    .Select(i => A_TMB[i, 0] * 0.5 + A_TMB[i, 1] * 0.5).ToArray();

static string Bools(bool[] v) => "[" + string.Join(", ", v.Select(b => b ? "True" : "False")) + "]";

Console.WriteLine("Meilleure reponse a une croyance PURE  : " + Bools(pureTmb));
Console.WriteLine("Meilleure reponse a une croyance MIXTE : " + Bools(mixteTmb));
Console.WriteLine("Gains de la Ligne sur p = (1/2, 1/2)   : ["
    + string.Join(", ", gainsMilieu.Select(v => v.ToString("F1", INV))) + "]");
Console.WriteLine("Reference LP (simplex maison)          : "
    + Bools(Enumerable.Range(0, 3).Select(i => MeilleureReponseMelangeLp(A_TMB, i)).ToArray()));

var (poncL, poncC) = StrategiesRationalisablesPures(A_TMB, B_TMB, new[] { "R0", "R1", "R2" }, new[] { "C0", "C1" });
var (ratL, ratC) = StrategiesRationalisables(A_TMB, B_TMB, new[] { "R0", "R1", "R2" }, new[] { "C0", "C1" });
Console.WriteLine("Rationalisables ponctuels (croyance pure) : lignes [" + string.Join(", ", poncL)
                  + "] | colonnes [" + string.Join(", ", poncC) + "]");
Console.WriteLine("Rationalisables (croyance mixte) : lignes [" + string.Join(", ", ratL)
                  + "] | colonnes [" + string.Join(", ", ratC) + "]");
Console.WriteLine("Equilibres de Nash purs         : ["
    + string.Join(", ", FindPureNash(jeuTmb).Select(p => "(" + jeuTmb.RowLabels[p.i] + ", " + jeuTmb.ColLabels[p.j] + ")")) + "]");


Trois lignes, deux colonnes
               C0          C1
-----------------------------
R0  ( 3.0, 1.0)  ( 0.0, 1.0)  
R1  ( 0.0, 1.0)  ( 3.0, 1.0)  
R2  ( 2.0, 0.0)  ( 2.0, 0.0)  


Meilleure reponse a une croyance PURE  : [True, True, False]


Meilleure reponse a une croyance MIXTE : [True, True, True]


Gains de la Ligne sur p = (1/2, 1/2)   : [1.5, 1.5, 2.0]


Reference LP (simplex maison)          : [True, True, True]


Rationalisables ponctuels (croyance pure) : lignes [R0, R1] | colonnes [C0, C1]


Rationalisables (croyance mixte) : lignes [R0, R1, R2] | colonnes [C0, C1]


Equilibres de Nash purs         : [(R0, C0), (R1, C1)]


In [9]:
// Confrontation des deux methodes exactes : l'enumeration des sommets et le
// simplex maison doivent rendre le meme verdict partout. Les sous-jeux sont
// confrontes aussi : c'est la que se joue le cas "croyance unique" (n = 1),
// que le jeu complet ne fait jamais apparaitre.
var rngC = new Rng(0);
int accords = 0, ecarts = 0;
for (int t = 0; t < 120; t++)
{
    var Ae = new double[2, 3];
    var Be = new double[2, 3];
    for (int u = 0; u < 2; u++) for (int v = 0; v < 3; v++)
    {
        Ae[u, v] = rngC.NextInt(0, 2);
        Be[u, v] = rngC.NextInt(0, 2);
    }
    var sousJeux = new List<double[,]>
    {
        Ae, Transpose(Be),
        SubMatrix(Ae, new List<int> { 0, 1 }, new List<int> { 0 }),
        SubMatrix(Ae, new List<int> { 0, 1 }, new List<int> { 0, 1 }),
        SubMatrix(Transpose(Be), new List<int> { 0, 1, 2 }, new List<int> { 0, 1 }),
    };
    foreach (var g in sousJeux)
        for (int i = 0; i < g.GetLength(0); i++)
        {
            bool r1 = MeilleureReponseMelange(g, i);
            bool r2 = MeilleureReponseMelangeLp(g, i);
            if (r1 == r2) accords++;
            else ecarts++;
        }
}
Console.WriteLine("Confrontation enumeration / LP : " + accords + " accords, " + ecarts + " ecarts");

// Confrontation sur des jeux a QUATRE colonnes : l'enumeration doit rester
// exacte quand le jeu a plus de trois colonnes -- c'est la qu'un sommet peut
// avoir trois frontieres actives, qu'une simple intersection de deux frontieres
// ne trouverait pas.
var rng4 = new Rng(1);
int accords4 = 0, ecarts4 = 0;
for (int t = 0; t < 500; t++)
{
    var A4 = new double[4, 4];
    var B4 = new double[4, 4];
    for (int u = 0; u < 4; u++) for (int v = 0; v < 4; v++)
    {
        A4[u, v] = rng4.NextInt(0, 5);
        B4[u, v] = rng4.NextInt(0, 5);
    }
    foreach (var g in new[] { A4, Transpose(B4) })
        for (int i = 0; i < 4; i++)
        {
            bool r1 = MeilleureReponseMelange(g, i);
            bool r2 = MeilleureReponseMelangeLp(g, i);
            if (r1 == r2) accords4++;
            else ecarts4++;
        }
}
Console.WriteLine("Confrontation 4 colonnes (500 jeux) : " + accords4 + " accords, " + ecarts4 + " ecarts");

// Temoin : un jeu 4x4 sur lequel une enumeration par intersections de deux
// frontieres SEULE se trompait (la derniere ligne etait declaree non
// rationalisable a tort) -- le jumeau Python le garde comme temoin de regression.
double[,] temoin = { { 4, 1, 0, 4 }, { 3, 3, 4, 1 }, { 4, 3, 3, 1 }, { 1, 1, 3, 2 } };
Console.WriteLine("Temoin 4x4 : enumeration " + Bools(Enumerable.Range(0, 4).Select(i => MeilleureReponseMelange(temoin, i)).ToArray())
    + " | LP " + Bools(Enumerable.Range(0, 4).Select(i => MeilleureReponseMelangeLp(temoin, i)).ToArray()));

Confrontation enumeration / LP : 1440 accords, 0 ecarts


Confrontation 4 colonnes (500 jeux) : 4000 accords, 0 ecarts


Temoin 4x4 : enumeration [True, True, True, True] | LP [True, True, True, True]


### Lecture du résultat : les deux notions se séparent, et voici pourquoi

Les lignes à lire ensemble disent la même chose que la section précédente, mais **mesurée** :

- **`Meilleure reponse a une croyance PURE`** : `R0` et `R1` y sont (chacune est la meilleure réponse à une colonne où elle vaut `3`), `R2` **non** — battue sur `C0` par `R0` et sur `C1` par `R1` ;
- **`Meilleure reponse a une croyance MIXTE`** : `R2` **y est**, comme `R0` et `R1` ;
- **`Gains de la Ligne sur p = (1/2, 1/2)`** : `1.5`, `1.5`, `2.0` — R2 l'emporte parce que la croyance mixte ne fait pas assez confiance à une colonne pour qu'on la batte ;
- la **référence par programme linéaire** — ici le **simplex maison**, pas `scipy` — rend le même verdict que l'énumération des sommets : les deux méthodes exactes concordent, y compris sur les sous-jeux qui font apparaître le cas « croyance unique » ;
- la **confrontation à quatre colonnes** confirme que l'énumération reste exacte quand le jeu a plus de trois colonnes : 500 jeux 4×4 tirés (graine `1`), 4 000 comparaisons avec la référence LP, **zéro désaccord**. L'énumération porte sur les intersections de `d = n - 1` frontières actives, et non sur deux frontières seulement.

Les deux lignes suivantes comparent les **survivants** des deux procédures d'élimination sur ce même jeu :

- le filtre **ponctuel** (croyance pure) élimine `R2` : elle ne survit qu'à une stratégie **pure** ;
- la **rationalisabilité** (croyance mixte) conserve `R2` : elle survit à une **distribution** de probabilités.

Et les équilibres de Nash du jeu sont `(R0, C0)` et `(R1, C1)` : ils sont rationalisables, comme l'inclusion l'exige.

Le message de la section tient donc en une phrase : **une stratégie peut être irrationnelle au sens de la croyance pure et rationnelle au sens de la croyance mixte**. Éliminer sur le critère « meilleure réponse à une stratégie pure » revient à écarter des stratégies qui ne sont pas irrationnelles — c'est un filtre plus fort, et il ne répond pas à la question que posent Bernheim et Pearce.

## 5. `Nash ⊆ rationalisable` — la mesure exhaustive

L'inclusion se comprend : dans un équilibre de Nash, la stratégie de chaque joueur est la meilleure réponse à celle des autres ; elle est donc la meilleure réponse à une stratégie **pure**, donc à une croyance **mixte** dégénérée, et l'élimination ne la retire jamais.

Mais cette inclusion a un sens **plus fort** qu'elle n'en a l'air : elle doit survivre à toutes les éliminations, y compris celles rendues possibles par les éliminations précédentes. On la vérifie donc sur la même famille exhaustive de 4096 jeux : pour chaque jeu, on compare les équilibres de Nash purs aux profils rationalisables.

On compare au passage les **trois filtres** côte à côte : l'IESDS (éliminer ce qui est *dominé*), l'admissibilité (éliminer ce qui n'est *jamais meilleure réponse à une croyance pure*) et la rationalisabilité (éliminer ce qui n'est *jamais meilleure réponse à une croyance mixte*). L'inclusion attendue est `Rationalisable ⊆ survivants de l'IESDS` — une stratégie strictement dominée n'est jamais une meilleure réponse, elle tombe dès le premier tour. On mesure aussi si les deux filtres de meilleure réponse se distinguent **sur cette famille** : c'est une question de fait, pas de théorème.

In [10]:
// Verification exhaustive : Nash <= rationalisable, et comparaison des filtres.
int violationsNash = 0, violationsInclusion = 0, divergencesIesds = 0;
int profilsNashTotal = 0, profilsRatTotal = 0, profilsAdmTotal = 0;
int jeuxAvecNash = 0, jeuxFiltresIdentiques = 0;
int nbJeuxAnalyses = 0;

foreach (var (A, B) in TousLesJeux2x3())
{
    nbJeuxAnalyses++;
    var jeu = new Game(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" });
    var nash = FindPureNash(jeu);
    var (ratL2, ratC2) = StrategiesRationalisables(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" });
    var rationalisables = ratL2.SelectMany(l => ratC2.Select(c => (l, c))).ToHashSet();
    var (admL2, admC2) = StrategiesRationalisablesPures(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" });
    var admissibles = admL2.SelectMany(l => admC2.Select(c => (l, c))).ToHashSet();

    profilsNashTotal += nash.Count;
    profilsRatTotal += rationalisables.Count;
    profilsAdmTotal += admissibles.Count;
    if (nash.Count > 0) jeuxAvecNash++;
    foreach (var (i, j) in nash)
        if (!rationalisables.Contains(("R" + i, "C" + j))) violationsNash++;

    // Les deux filtres de meilleure reponse se distinguent-ils sur CETTE famille ?
    if (new HashSet<string>(ratL2).SetEquals(admL2) && new HashSet<string>(ratC2).SetEquals(admC2))
        jeuxFiltresIdentiques++;

    // Comparaison avec l'IESDS : la rationalisabilite est-elle incluse ?
    var (iesdsL, iesdsC, _, _, _) = Iesds(A, B, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" });
    if (!new HashSet<string>(ratL2).IsSubsetOf(new HashSet<string>(iesdsL))
        || !new HashSet<string>(ratC2).IsSubsetOf(new HashSet<string>(iesdsC)))
        violationsInclusion++;
    // Et l'inverse : l'IESDS garde-t-elle des strategies que la rationalisabilite elimine ?
    if (!new HashSet<string>(iesdsL).IsSubsetOf(new HashSet<string>(ratL2))
        || !new HashSet<string>(iesdsC).IsSubsetOf(new HashSet<string>(ratC2)))
        divergencesIesds++;
}

Console.WriteLine("Jeux analyses                                 : " + nbJeuxAnalyses);
Console.WriteLine("Jeux avec au moins un equilibre de Nash       : " + jeuxAvecNash);
Console.WriteLine("Profils de Nash rencontres au total          : " + profilsNashTotal);
Console.WriteLine("Profils rationalisables rencontres           : " + profilsRatTotal);
Console.WriteLine("Profils rationalisables ponctuels rencontres : " + profilsAdmTotal);
Console.WriteLine("Jeux ou les deux filtres coincident          : " + jeuxFiltresIdentiques);
Console.WriteLine("Contre-exemples a Nash <= rationalisable     : " + violationsNash);
Console.WriteLine("Contre-exemples a Rationalisable <= IESDS     : " + violationsInclusion);
Console.WriteLine("Jeux ou l'IESDS garde plus que la rationalisabilite : " + divergencesIesds);

Jeux analyses                                 : 4096


Jeux avec au moins un equilibre de Nash       : 4066


Profils de Nash rencontres au total          : 11520


Profils rationalisables rencontres           : 20286


Profils rationalisables ponctuels rencontres : 20286


Jeux ou les deux filtres coincident          : 4096


Contre-exemples a Nash <= rationalisable     : 0


Contre-exemples a Rationalisable <= IESDS     : 0


Jeux ou l'IESDS garde plus que la rationalisabilite : 372


### Lecture du résultat : ce que la famille exhaustive dit des trois filtres

Les compteurs se lisent en quatre temps.

**1. `Nash ⊆ rationalisable` tient.** Le compteur de contre-exemples est **zéro** : sur les 4096 jeux exhaustivement énumérés, chaque profil de Nash est bien un profil rationalisable. Ce n'est donc pas une intuition de manuel — elle tient sur toute la famille, et sa raison est désormais explicite : une stratégie d'un équilibre de Nash est la meilleure réponse à une stratégie **pure** de l'adversaire, donc à une croyance mixte dégénérée.

**2. La divergence est de taille.** 20286 profils rationalisables contre 11520 profils de Nash sur la même famille : la rationalisabilité est strictement plus permissive. Autrement dit, la rationalisabilité est une **approximation** de l'équilibre de Nash — plus facile à vérifier, plus robuste, mais elle ne dit pas quel sera l'équilibre. C'est le sens de l'inclusion :

$$\text{Nash} \subsetneq \text{Rationalisable}$$

**3. Sur cette famille, les deux filtres de meilleure réponse coïncident — et ce n'est pas un théorème.** Le compteur `Profils rationalisables ponctuels` vaut le même total que `Profils rationalisables`, et le compteur `Jeux ou les deux filtres coincident` est de **4096 sur 4096** : sur les jeux 2×3 à gains dans `{0, 1}`, « être la meilleure réponse à une stratégie pure » et « être la meilleure réponse à une croyance mixte » donnent le même ensemble de survivants, jeu après jeu.

C'est un **fait de famille, pas une identité** : la section 4 a mesuré un jeu où les deux notions divergent (`R2` disparaît du filtre ponctuel et survit à la rationalisabilité). La famille 2×3 à gains binaires est trop pauvre pour faire apparaître la différence — il fallait un jeu à trois lignes, dont deux se partagent l'avantage.

**4. Les deux filtres ne sont pas le même filtre — et l'inclusion va dans le sens inverse de ce qu'on attendait.** Le compteur `Rationalisable <= IESDS` est **zéro** : la rationalisabilité ne conserve jamais une stratégie que l'IESDS aurait éliminée (une stratégie strictement dominée n'est jamais une meilleure réponse, elle tombe donc dès le premier tour). Mais le dernier compteur est **non nul** : dans 372 jeux sur 4096, la rationalisabilité élimine une stratégie que la dominance stricte avait conservée.

Autrement dit, **l'IESDS est plus grossière que la rationalisabilité**. Les deux procédures ne sont pas emboîtées l'une dans l'autre : l'IESDS répond à « qui est *dominé* ? » — et il ne teste qu'une stratégie pure comme dominateur —, la rationalisabilité à « qui n'est *jamais optimal* ? », toutes mélangeologies admises. Une stratégie peut être non dominée par une stratégie pure et néanmoins ne jamais être une meilleure réponse : la rationalisabilité la retire, l'IESDS la garde.

C'est la raison mathématique de l'inclusion `Nash ⊆ rationalisable` : seul le filtre par optimalité est **garanti** de conserver tous les équilibres, pas le filtre par dominance.

In [11]:
// Le lemme de Pearce, verifie par echantillon. En deux joueurs, l'elimination
// iteree des strategies "jamais meilleure reponse a une croyance mixte" doit
// coincider avec l'elimination iteree des strategies strictement dominees par
// un MELANGE. On le verifie en confrontant les deux procedures sur un
// echantillon de la famille.
//
// ATTENTION : la rationalisabilite (croyance mixte) est calculee ici par
// ENUMERATION DES SOMMETS (rapide) ; la dominance stricte mixte par PROGRAMME
// LINEAIRE (lent). L'echantillon est donc borne -- la verification est un
// echantillon, pas un resultat exhaustif.
var rngLemma = new Rng(0);
int jeuxLemma = 400;
int ecartsLemma = 0;
var ecartsDetail = new List<string>();
for (int t = 0; t < jeuxLemma; t++)
{
    var Alem = new double[2, 3];
    var Blem = new double[2, 3];
    for (int u = 0; u < 2; u++) for (int v = 0; v < 3; v++)
    {
        Alem[u, v] = rngLemma.NextInt(0, 2);
        Blem[u, v] = rngLemma.NextInt(0, 2);
    }

    var (rl, rc) = StrategiesRationalisables(Alem, Blem, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" });
    var (dl, dc) = EliminationDominanceMixte(Alem, Blem, new[] { "R0", "R1" }, new[] { "C0", "C1", "C2" });

    if (!new HashSet<string>(rl).SetEquals(dl) || !new HashSet<string>(rc).SetEquals(dc))
    {
        ecartsLemma++;
        if (ecartsDetail.Count < 3)
            ecartsDetail.Add("  ECART : A=[" + string.Join(",", Alem.Cast<double>().Select(v => v.ToString("0", INV))) + "] B=["
                + string.Join(",", Blem.Cast<double>().Select(v => v.ToString("0", INV))) + "] | rat [" + string.Join(",", rl)
                + "] / [" + string.Join(",", rc) + "] | dom [" + string.Join(",", dl) + "] / [" + string.Join(",", dc) + "]");
    }
}

Console.WriteLine("Echantillon du lemme de Pearce (PRNG maison, graine 0) : " + jeuxLemma + " jeux");
Console.WriteLine("Jeux ou les deux procedures divergent      : " + ecartsLemma);
foreach (var d in ecartsDetail) Console.WriteLine(d);

Echantillon du lemme de Pearce (PRNG maison, graine 0) : 400 jeux


Jeux ou les deux procedures divergent      : 0


### Lecture du résultat : le lemme de Pearce, et ce qu'il ne dit pas

Les deux lignes disent la même chose : sur les **400** jeux tirés dans la famille (graine `0`, reproductible), les deux procédures — élimination des stratégies *jamais meilleure réponse à une croyance mixte*, et élimination des stratégies *strictement dominées par un mélange* — donnent **le même ensemble de survivants**, et le compteur de divergences est **zéro**.

C'est le lemme de Pearce : en deux joueurs, les deux filtres sont le même filtre écrit autrement. Il est important de ne pas le confondre avec l'IESDS du carnet, qui teste un dominateur **pur** : c'est précisément pour cela que l'IESDS garde, dans 372 jeux sur 4096, une stratégie que la rationalisabilité retire.

Deux limites à garder en tête :

- la rationalisabilité est calculée par **énumération des sommets** (rapide) et la dominance stricte mixte par **programme linéaire** — ici le simplex maison, comme la référence du jumeau Python : la confrontation est donc bornée à un **échantillon de 400 jeux**, pas aux 4096 jeux de la famille. C'est une vérification par échantillon, pas une preuve ;
- le lemme est un fait de **deux joueurs** : il ne s'étend pas tel quel aux jeux à trois joueurs et plus, où l'argument de la section 4 ne joue plus du tout.

## 6. Exercices

Quatre exercices pour mettre en pratique les outils du carnet. Les solutions ne sont pas fournies : chaque cellule de code est un squelette à compléter (`// TODO`). Toutes les briques nécessaires (`Game`, `FindPureNash`, `DominanceStricteLignes`, `DominanceStricteColonnes`, `Iesds`, `IesdsFaible`, `StrategiesRationalisablesPures` — alias historique `StrategiesAdmissibles` —, `StrategiesRationalisables`, `MeilleureReponseMelange`, `MeilleureReponseMelangeLp`, `StrictementDominanteMixte`, `EliminationDominanceMixte`, `TousLesJeux2x3`) restent disponibles dans l'environnement du carnet.

### Exercice 1 : l'IESDS pas-à-pas sur un jeu 3×3

Reprenez le jeu en cascade du carnet, mais avec des gains **différents** : la dynamique doit être la même — une stratégie de la Colonne ne devient dominée qu'après le retrait d'une stratégie de la Ligne.

| | C0 | C1 | C2 |
|---|---|---|---|
| **R0** | (0, 0) | (0, 0) | (0, 0) |
| **R1** | (2, 3) | (2, 2) | (2, 2) |
| **R2** | (4, 0) | (1, 4) | (4, -2) |

**Objectif** : écrire la fonction `Iesds3x3` qui élimine les stratégies strictement dominées une par une et renvoie le **journal** des éliminations.

- **Indice** : `DominanceStricteLignes` et `DominanceStricteColonnes` opèrent sur une sous-matrice — il faut les rejouer sur le jeu réduit, à chaque étape.
- **Etape 1** : trouver la première stratégie strictement dominée (les lignes d'abord).
- **Etape 2** : la retirer, puis recommencer sur le jeu réduit.
- **Etape 3** : renvoyer `(survivants_lignes, survivants_colonnes, journal)` et l'afficher.

In [12]:
// Exercice 1 : l'IESDS pas-a-pas sur un jeu 3x3
// ==================================================================
// TODO etudiant : completer Iesds3x3 (elimination + journal).
// TODO etudiant : renvoyer (survivants_lignes, survivants_colonnes, journal)
object resultatEx1 = null;  // TODO etudiant

Console.WriteLine("Exercice 1 a completer : la cascade attendue comporte deux eliminations.");

Exercice 1 a completer : la cascade attendue comporte deux eliminations.


### Exercice 2 : l'IESDS dépend-elle de l'ordre ?

Reprenez le jeu en cascade du carnet (`A_cascade`, `B_cascade`) et vérifiez **explicitement** que les survivants ne dépendent pas de l'ordre d'élimination.

**Objectif** : comparer les survivants obtenus sous deux ordres différents, et vérifier qu'ils coïncident.

- **Indice** : la fonction `Iesds` accepte un paramètre `ordre`, liste de couples `('L', index)` ou `('C', index)` aux indices **originaux**.
- **Etape 1** : construire un ordre où la Colonne est examinée avant la Ligne, et un autre où la Ligne est examinée avant la Colonne.
- **Etape 2** : comparer les deux résultats et afficher un message explicite si (et seulement si) ils diffèrent.
- **Etape 3** : faire de même avec `IesdsFaible` (dominance **faible**) sur le même jeu, et dire si la conclusion change.

In [13]:
// Exercice 2 : l'IESDS depend-elle de l'ordre ?
// ==================================================================
// TODO etudiant : comparer les survivants de Iesds sous deux ordres opposes.
// TODO etudiant : afficher "ordre independant" ou "ordre dependant"
object conclusionEx2 = null;  // TODO etudiant

Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : stratégies rationalisables et croyance mixte

Considérez le jeu suivant (gains de la Ligne, gains de la Colonne) :

| | C0 | C1 |
|---|---|---|
| **R0** | (2, 0) | (0, 1) |
| **R1** | (1, 0) | (1, 0) |

**Objectif** : calculer les stratégies **rationalisables** de ce jeu et les comparer à ses équilibres de Nash.

- **Indice** : `StrategiesRationalisables(A, B, labelsL, labelsC)` renvoie les survivants de la rationalisabilité (meilleure réponse à une croyance **mixte**) ; `StrategiesRationalisablesPures(A, B, labelsL, labelsC)` renvoie ceux du filtre **ponctuel** (meilleure réponse à une croyance **pure**) ; `FindPureNash(new Game(A, B, ...))` renvoie les équilibres purs.
- **Etape 1** : calculer les deux ensembles de survivants et les équilibres de Nash.
- **Etape 2** : vérifier que tout profil de Nash est rationalisable.
- **Etape 3** : afficher les profils rationalisables qui ne sont pas des équilibres de Nash (il y en a au moins un).
- **Etape 4** : sur un jeu **à trois lignes** que vous choisissez, exhibez une stratégie qui survit à la rationalisabilité mais pas au filtre ponctuel — c'est le contre-exemple de la section 4, à refaire par vous-même.

In [14]:
// Exercice 3 : strategies rationalisables
// ==================================================================
double[,] A_ex3 = { { 2, 0 }, { 1, 1 } };
double[,] B_ex3 = { { 0, 1 }, { 0, 0 } };

// TODO etudiant : calculer les strategies rationalisables (croyance MIXTE) ET
// les strategies admissibles (croyance PURE), puis les equilibres de Nash purs.
// TODO etudiant : afficher les profils rationalisables qui ne sont PAS de Nash
// TODO etudiant : trouver une strategie (sur un jeu a trois lignes) qui
//                 survit a la rationalisabilite mais pas a l'admissibilite
object divergenceEx3 = null;  // TODO etudiant

Console.WriteLine("Exercice 3 a completer");

Exercice 3 a completer


### Exercice 4 : chercher un contre-exemple en dominance faible

La dominance faible rend l'élimination dépendante de l'ordre (mesuré dans le carnet). Votre travail est de **retrouver** un contre-exemple par vous-même, dans une famille que vous choisissez.

**Objectif** : écrire une fonction `ChercherContreExempleFaible` qui parcourt une famille de jeux et renvoie le premier jeu dont les survivants diffèrent selon l'ordre.

- **Indice** : `IesdsFaible` accepte lui aussi un paramètre `ordre` ; `TousLesJeux2x3` fournit une famille exhaustive, mais vous pouvez la réduire (par exemple en fixant `A` et en balayant `B`).
- **Etape 1** : choisir la famille à explorer.
- **Etape 2** : comparer les survivants sous deux ordres opposés.
- **Etape 3** : renvoyer `(A, B, survivants_ordre_1, survivants_ordre_2)` et afficher le jeu trouvé.

In [15]:
// Exercice 4 : chercher un contre-exemple en dominance faible
// ==================================================================
object ChercherContreExempleFaible()
{
    // Renvoie (A, B, survivants_ordre_1, survivants_ordre_2) ou null.
    // TODO etudiant
    return null;
}

object contreExempleEx4 = null;  // TODO etudiant

Console.WriteLine("Exercice 4 a completer");

Exercice 4 a completer


## 7. Résumé

| Concept | Définition | Propriété mesurée |
|---|---|---|
| **Dominance stricte** | $u_i(s_i, s_{-i}) > u_i(s_i', s_{-i})$ pour tout $s_{-i}$ | — |
| **IESDS** | élimination itérée des stratégies strictement dominées (par une stratégie **pure**) | **indépendante de l'ordre** (4096 jeux 2×3, 10 ordres chacun) ; garde **plus** de stratégies que la rationalisabilité (372 jeux sur 4096) |
| **Dominance faible** | $\geq$ partout, $>$ quelque part | **dépendante de l'ordre** (contre-exemple mesuré) |
| **Rationalisabilité ponctuelle** | la stratégie est meilleure réponse à au moins une croyance **pure** survivante | filtre **plus fort** que la rationalisabilité ; n'élimine jamais un équilibre de Nash ; sur la famille 2×3 mesurée, identique à la rationalisabilité |
| **Rationalisabilité** | la stratégie est meilleure réponse à au moins une croyance **mixte** survivante | n'élimine jamais un équilibre de Nash ; coïncide avec l'élimination de la dominance stricte **mixte** (échantillon mesuré) |
| **Profil rationalisable** | produit de stratégies rationalisables | **strictement plus nombreux** que les profils de Nash |

Les deux inclusions à retenir :

$$\text{Nash} \subsetneq \text{Rationalisable} \subseteq \text{IESDS (survivants)}$$

`Nash ⊊ Rationalisable` : 20286 profils rationalisables contre 11520 profils de Nash sur la famille exhaustive, et le profil `(Opera, Ballet)` de la Bataille des Sexes survit au filtre alors qu'il n'est pas un équilibre.

**Parité .NET ⇄ Python.** Ce jumeau reproduit les mesures du jumeau Python avec trois briques réécrites from scratch : l'énumération exhaustive des 4096 jeux (même famille), l'énumération des sommets du polyèdre (même géométrie, `d = n - 1` frontières actives), et un **simplex à deux phases** qui remplace `scipy.optimize.linprog` — confronté à l'énumération sur 5 440 instances (1 440 + 4 000) sans un seul désaccord. Seuls les tirages aléatoires diffèrent (PRNG mulberry32 maison au lieu de `numpy.random`), pour les mêmes graines — les compteurs déterministes (4096, 1536, 4066, 11520, 20286, 372, et tous les zéros) sont identiques.

***

**Navigation** : [<< 2-NormalForm](GameTheory-02-NormalForm-Python.ipynb) · [2b-Lean-Definitions](GameTheory-02b-Lean-Definitions-Lean.ipynb) · [2e-Python (jumeau)](GameTheory-02e-IESDS-Rationalisation-Python.ipynb) | [Index](README.md) | [4-NashEquilibrium >>](GameTheory-04-NashEquilibrium-Python.ipynb)